# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Binary classification.**

My lane is Content Refresh Prioritization — deciding *which* content items to review first out of
thousands. That's not clustering (I'm not grouping unlabeled items into unknown segments), not
ranking in the learning-to-rank sense (I don't have relevance-judged query-document pairs), and
not plain regression (I don't need a precise continuous number, I need a yes/no triage bucket:
"is this item declining, review it or not"). A classifier that outputs `is_declining_proxy`
(0/1) plus a probability score is the natural fit — the probability doubles as a ranking signal
for the queue, so classification quietly subsumes the ranking need here too.


In [ ]:
# Quick sanity check: this lane is really a classification problem because the
# thing I'd predict is a two-valued bucket, not a continuous number or an unlabeled group.
print("Task type: binary classification")
print("Positive class (1): is_declining_proxy == 1 -> flag for review")
print("Negative class (0): is_declining_proxy == 0 -> leave alone this cycle")


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_proxy` — a proxy, not a directly observed outcome.**

Defined as: 1 if a content item's GSC impressions in the second half of March 2026 are more than
20% lower than the first half, else 0 (same definition used in the ML-04 data contract, so the
two assignments stay consistent).

It's a *proxy* label, not a ground-truth outcome, for two reasons: (1) it's a same-month,
same-window comparison rather than a true future prediction (predicting April from March would
be the stronger, capstone-grade version), and (2) "impressions dropped 20%" is a rule I chose,
not something FlyRank observed and recorded as ground truth — a different threshold (10%, 30%)
would relabel some items differently. I'm explicit about this in every claim so a reviewer never
mistakes the proxy for a real observed decline.


In [ ]:
# The label is a RULE applied to observed data, not an observed outcome itself —
# showing the rule as code makes that explicit instead of leaving it as a claim.
def is_declining_proxy(imp_first_half, imp_second_half, threshold=0.8):
    """1 if second-half impressions fall below `threshold` x first-half impressions."""
    return int(imp_second_half < threshold * imp_first_half)

# example: a page with 1000 -> 750 impressions is flagged; 1000 -> 850 is not
print(is_declining_proxy(1000, 750))   # 1  (25% drop, flagged)
print(is_declining_proxy(1000, 850))   # 0  (15% drop, not flagged)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50 (top 50 flagged items), not raw accuracy or AUC alone.**

The business action here is limited: a content team can only actually review N items per cycle.
So the metric that matters isn't "how good is the model on average" (accuracy/AUC blur this) —
it's "of the top 50 items the model tells us to look at first, how many are genuinely declining."
That's the same Precision@K framing the starter pipeline itself uses
(Precision@50 ≈ 0.24 → 0.74, rule vs. model). I'd still report AUC as a secondary,
threshold-free sanity check, but Precision@K is the number I'd defend to a non-technical
stakeholder, because it maps directly to "was reviewing these 50 pages worth the team's time."



In [ ]:
# Precision@K, defined in code so the metric claim isn't just a word choice.
import numpy as np

def precision_at_k(y_true, y_score, k=50):
    order = np.argsort(-np.asarray(y_score))[:k]
    return np.asarray(y_true)[order].mean()

# toy example: 5 of the top 10 scored items are true positives -> Precision@10 = 0.5
y_true  = [1,0,1,0,1,0,1,0,1,0]
y_score = [9,1,8,2,7,3,6,4,5,0]
print(f"Precision@10 example: {precision_at_k(y_true, y_score, k=10):.2f}")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item's GSC performance for one client, for the first half of
March 2026** — the same grain as the ML-04 data contract
(`report_date + client_hash_id + content_hash_id`, aggregated to `content_hash_id + client_hash_id`
once the first-half window is fixed).

I load it the same way as ML-04: DuckDB reading the `month=2026-03` partition straight from the
`FlyRank/internship-warehouse` dataset on Hugging Face (gated, needs an accepted-terms HF token).
**Note:** this cell needs to run in Colab with `HF_TOKEN` set as a secret — it can't execute in a
sandboxed offline environment. Run it there, then this cell's printed `.head()` becomes the real
evidence for this section.


In [ ]:
%pip -q install duckdb
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

unit_of_analysis = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_first_half,
           SUM(gsc_clicks)      AS clicks_first_half
    FROM {fact_march}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

print(f"{len(unit_of_analysis):,} rows -> one row = one content item x one client, first-half March")
unit_of_analysis.head()


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A single if-statement can't hold the interaction between impressions, CTR, average
position, and consistency (active days) at once.** A hand rule like "flag if position > 15"
mislabels a page with high impressions but a slightly worse position, and misses a page whose
position is fine but whose CTR and active-day count are both quietly collapsing. The *combination*
of weak signals — not any single one crossing a threshold — is what actually predicts decline,
and finding the right weights for that combination by hand, across thousands of items with
different baselines, isn't realistic. That's exactly the "hand-rule vs. learned-weights" gap the
starter pipeline demonstrates numerically (Precision@50 rule ≈ 0.24 vs. model ≈ 0.74).


In [ ]:
# Illustrating the point with a tiny synthetic example: no single threshold on ONE
# feature separates the two classes cleanly, but a combination does.
import numpy as np
rng = np.random.default_rng(0)

declining     = rng.normal(loc=[8, 0.02], scale=[3, 0.01], size=(50, 2))   # [position, ctr]
not_declining = rng.normal(loc=[9, 0.025], scale=[4, 0.01], size=(50, 2))  # overlapping on purpose

# a single-feature rule on position alone can't cleanly separate these -- ranges overlap heavily
print("declining position range:    ", declining[:,0].min().round(1), "-", declining[:,0].max().round(1))
print("not-declining position range:", not_declining[:,0].min().round(1), "-", not_declining[:,0].max().round(1))
print("-> ranges overlap, so no single position threshold works -- this is why a model that")
print("   weighs several features together beats one if-statement.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.